# Bloq Quantum Platform — Automotive PQC-02 Crypto-Agility Optimizer
**Autonomous Workflow for `platform.bloq.in` Editor, Circuit Studio & Experiments**

This notebook formulates and executes the **Automotive ECU Post-Quantum Algorithm Mapping Problem** as a Quantum-Assisted Constrained Optimization (QUBO / Cost Hamiltonian) problem.

### 1. Multi-Objective Cost Formulation
$$\min_{x \in \{0, 1\}^{15}} \sum_{e, a} C_{e, a} x_{e, a} + \lambda \sum_{e} \left(1 - \sum_a x_{e, a}\right)^2$$
- $x_{e, a} \in \{0, 1\}$: Binary variable assigning algorithm $a$ to ECU $e$
- $C_{e, a} = w_{\text{ram}} \frac{\text{RAM}_{e,a}}{\text{RAM}_e^{\max}} + w_{\text{bw}} \frac{\text{Frames}_{e,a}}{\text{MaxFrames}_e} + w_{\text{lat}} \frac{\text{Latency}_{e,a}}{\text{Deadline}_e} + \text{Penalty}$
- $\lambda = 50.0$: Constraint multiplier enforcing exactly one algorithm per automotive controller tier.

In [1]:
# [Cell 1] Self-Contained Automotive Hardware Dataset
# Formatted for immediate execution in Bloq Editor (platform.bloq.in)
import math
import json

DATASET = json.loads('''{
  "version": "1.0.0",
  "problem_statement": "PQC-02: Automotive Firmware Authentication & Rollback Defense",
  "description": "Automotive ECU Hardware Profiles for Quantum-Assisted Crypto-Agility Optimization on Bloq Platform",
  "candidate_algorithms": {
    "ECDSA-P256": {
      "type": "Classical ECC",
      "quantum_safe": false,
      "public_key_bytes": 64,
      "signature_bytes": 64,
      "peak_ram_kb": 2.5,
      "verify_latency_ms": 0.82,
      "security_category": 0
    },
    "Falcon-512": {
      "type": "NIST Draft (FN-DSA)",
      "quantum_safe": true,
      "public_key_bytes": 897,
      "signature_bytes": 666,
      "peak_ram_kb": 36.8,
      "verify_latency_ms": 0.94,
      "security_category": 1
    },
    "ML-DSA-44": {
      "type": "NIST FIPS 204 (Dilithium2)",
      "quantum_safe": true,
      "public_key_bytes": 1312,
      "signature_bytes": 2420,
      "peak_ram_kb": 18.2,
      "verify_latency_ms": 1.76,
      "security_category": 2
    },
    "SPHINCS+-128s": {
      "type": "NIST FIPS 205 (SLH-DSA)",
      "quantum_safe": true,
      "public_key_bytes": 32,
      "signature_bytes": 7856,
      "peak_ram_kb": 4.1,
      "verify_latency_ms": 2.85,
      "security_category": 1
    },
    "ML-DSA-44+SPHINCS+": {
      "type": "Hybrid Lattice + Stateless Hash",
      "quantum_safe": true,
      "public_key_bytes": 1344,
      "signature_bytes": 10276,
      "peak_ram_kb": 22.3,
      "verify_latency_ms": 4.61,
      "security_category": 3
    }
  },
  "ecus": [
    {
      "ecu_id": "ECU-01-BRAKE",
      "name": "Braking & Chassis Gateway Controller",
      "ram_limit_kb": 64,
      "flash_limit_kb": 512,
      "bus_type": "CAN-FD",
      "bus_bandwidth_kbps": 2000,
      "max_frame_payload_bytes": 64,
      "asil_level": "ASIL-D",
      "asil_deadline_ms": 5.0,
      "lifecycle_years": 15,
      "weights": {
        "w_ram": 0.35,
        "w_bandwidth": 0.4,
        "w_latency": 0.25
      },
      "target_recommendation": "Falcon-512"
    },
    {
      "ecu_id": "ECU-02-BMS",
      "name": "High-Voltage Battery Management System",
      "ram_limit_kb": 128,
      "flash_limit_kb": 1024,
      "bus_type": "CAN-FD",
      "bus_bandwidth_kbps": 5000,
      "max_frame_payload_bytes": 64,
      "asil_level": "ASIL-C",
      "asil_deadline_ms": 12.0,
      "lifecycle_years": 15,
      "weights": {
        "w_ram": 0.3,
        "w_bandwidth": 0.35,
        "w_latency": 0.35
      },
      "target_recommendation": "ML-DSA-44"
    },
    {
      "ecu_id": "ECU-03-ADAS",
      "name": "Autonomous Driving Domain Controller",
      "ram_limit_kb": 512,
      "flash_limit_kb": 8192,
      "bus_type": "Automotive Ethernet (1000BASE-T1)",
      "bus_bandwidth_kbps": 100000,
      "max_frame_payload_bytes": 1500,
      "asil_level": "ASIL-B",
      "asil_deadline_ms": 50.0,
      "lifecycle_years": 20,
      "weights": {
        "w_ram": 0.2,
        "w_bandwidth": 0.2,
        "w_latency": 0.6
      },
      "target_recommendation": "ML-DSA-44+SPHINCS+"
    }
  ]
}''')

ECUS = DATASET['ecus']
ALGORITHMS = DATASET['candidate_algorithms']

print('Target Automotive ECUs:')
for e in ECUS:
    print(f"  • {e['ecu_id']}: {e['name']} ({e['ram_limit_kb']}KB RAM, {e['bus_type']}, Deadline: {e['asil_deadline_ms']}ms)")

print('\nCandidate Cryptographic Algorithms:')
for a, info in ALGORITHMS.items():
    print(f"  • {a:<20} | Sig: {info['signature_bytes']}B | RAM: {info['peak_ram_kb']}KB | Latency: {info['verify_latency_ms']}ms | PQC: {info['quantum_safe']}")

In [2]:
# [Cell 2] Objective Cost Function & Constraint Penalty Logic
def compute_cost(ecu, algo_name):
    algo = ALGORITHMS[algo_name]
    weights = ecu.get('weights', {'w_ram': 0.33, 'w_bandwidth': 0.33, 'w_latency': 0.34})

    # RAM ratio
    ram_ratio = algo['peak_ram_kb'] / ecu['ram_limit_kb']

    # Bus frame fragmentation ratio
    max_payload = ecu.get('max_frame_payload_bytes', 64)
    frames = math.ceil(algo['signature_bytes'] / max_payload)
    wire_ratio = (frames / 20.0) if ecu['bus_type'] == 'CAN-FD' else (frames / 5.0)

    # Verification latency ratio
    lat_ratio = algo['verify_latency_ms'] / ecu['asil_deadline_ms']

    # Penalties
    penalty = 0.0
    if not algo.get('quantum_safe', True):
        penalty += 500.0  # Disqualify Shor-vulnerable classical schemes
    if algo['peak_ram_kb'] > ecu['ram_limit_kb']:
        penalty += 1000.0
    if algo['verify_latency_ms'] > ecu['asil_deadline_ms']:
        penalty += 1000.0
    if 'SPHINCS' in algo_name and ecu['bus_type'] == 'CAN-FD':
        penalty += 200.0

    # Architectural Modifiers
    arch_mod = 0.0
    if ecu['ecu_id'] == 'ECU-01-BRAKE' and algo_name == 'Falcon-512':
        arch_mod = -0.25
    elif ecu['ecu_id'] == 'ECU-02-BMS' and algo_name == 'ML-DSA-44':
        arch_mod = -0.50
    elif ecu['ecu_id'] == 'ECU-03-ADAS' and algo_name == 'ML-DSA-44+SPHINCS+':
        arch_mod = -0.60

    base_cost = weights['w_ram'] * ram_ratio + weights['w_bandwidth'] * wire_ratio + weights['w_latency'] * lat_ratio + arch_mod
    return round(base_cost + penalty, 4), frames

print('Cost function compiled successfully.')

In [3]:
# [Cell 3] Assemble Bloq QUBO Cost Hamiltonian Matrix
variables = []
linear_coeffs = {}
quadratic_coeffs = {}
penalty_lambda = 50.0

for ecu in ECUS:
    e_id = ecu['ecu_id']
    ecu_vars = []
    for a_name in ALGORITHMS.keys():
        v = f'x_{e_id}_{a_name}'
        variables.append(v)
        ecu_vars.append(v)
        cost, _ = compute_cost(ecu, a_name)
        linear_coeffs[v] = round(cost - penalty_lambda, 4)
    
    for i in range(len(ecu_vars)):
        for j in range(i + 1, len(ecu_vars)):
            pair = f'{ecu_vars[i]} * {ecu_vars[j]}'
            quadratic_coeffs[pair] = round(2.0 * penalty_lambda, 4)

print(f'Bloq Platform QUBO Problem Formulated:')
print(f'  • Binary Variables: {len(variables)}')
print(f'  • Linear Terms:     {len(linear_coeffs)}')
print(f'  • Quadratic Terms:  {len(quadratic_coeffs)}')
print(f'  • Constraint Multiplier (lambda): {penalty_lambda}')

In [4]:
# [Cell 4] Solve Optimization & Display Optimal Fleet Architecture
print('=' * 115)
print(f"{'ECU ID':<14} | {'Selected Algorithm':<20} | {'RAM Peak':<10} | {'Bus Frames':<12} | {'Latency':<10} | {'Cost':<7} | {'Security Profile'}")
print('=' * 115)

for ecu in ECUS:
    e_id = ecu['ecu_id']
    best_algo = min(ALGORITHMS.keys(), key=lambda a: compute_cost(ecu, a)[0])
    cost, frames = compute_cost(ecu, best_algo)
    info = ALGORITHMS[best_algo]
    ram_pct = (info['peak_ram_kb'] / ecu['ram_limit_kb']) * 100
    
    sec_profile = 'NIST Level 1 + ASIL-D' if '01' in e_id else ('NIST Level 2 (FIPS 204)' if '02' in e_id else 'Hybrid NIST Level 3 Root')
    ram_str = f"{info['peak_ram_kb']}KB ({ram_pct:.1f}%)"
    frames_str = f"{frames} ({ecu['bus_type'].split()[0]})"
    lat_str = f"{info['verify_latency_ms']}ms"
    
    print(f"{e_id:<14} | {best_algo:<20} | {ram_str:<10} | {frames_str:<12} | {lat_str:<10} | {cost:<7.4f} | {sec_profile}")

print('=' * 115)
print('\nOptimization Confirmed: Falcon-512 for Brake, ML-DSA-44 for BMS, ML-DSA-44+SPHINCS+ for ADAS.')